In [ ]:
#| default_exp machine_learning.training_helpers

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-21. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

The training notebooks (in `../trouver_personal_playground/`) re-inline the same scaffolding with silent drift: `compute_metrics` in 9 notebooks, `preprocess_function` in 5, word-piece label alignment in 4. These are the shared, task-independent versions — parameterized instead of relying on notebook globals (`tokenizer`, `label_list`), and with the metric object injectable so tests run without `evaluate`/network downloads.

Heavy ML packages (`transformers`, `datasets`, `evaluate`) are **not** imported here: these functions operate on plain dicts, lists, and arrays.

In [ ]:
#| export
from os import PathLike
from pathlib import Path

import numpy as np

In [ ]:
#| export
def _aligned_label_ids(word_ids: list, labels: list, label_all_tokens: bool = True) -> list:
    """
    Map one sentence's word-level `labels` onto its subword `word_ids`.

    Special tokens map to `None` and get `-100` (ignored by the loss).
    Continuation subwords repeat their word's label when
    `label_all_tokens`, else also get `-100`.
    """
    aligned, previous_word_idx = [], None
    for word_idx in word_ids:
        if word_idx is None:
            aligned.append(-100)
        elif word_idx != previous_word_idx:
            aligned.append(labels[word_idx])
        else:
            aligned.append(labels[word_idx] if label_all_tokens else -100)
        previous_word_idx = word_idx
    return aligned

In [ ]:
#| export
def align_labels_to_tokens(examples: dict, tokenizer, label_all_tokens: bool = True) -> dict:
    """
    Tokenize a batch of NER examples and align BIO labels to subwords.

    The HF `datasets.map` preprocessing shared by every token-classification
    pipeline, with the tokenizer passed in instead of read from a notebook
    global. Labels for special tokens (and continuation subwords when
    `label_all_tokens=False`) become `-100`, the `CrossEntropyLoss` ignore index.

    **Parameters**
    - `examples` (*dict*) — Batch with `'tokens'` and `'ner_tags'` columns.
    - `tokenizer` — A fast (word-piece) HF tokenizer.
    - `label_all_tokens` (*bool*) — Label every subword of a word, not just its first.
    """
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)
    tokenized_inputs['labels'] = [
        _aligned_label_ids(tokenized_inputs.word_ids(batch_index=i), label, label_all_tokens)
        for i, label in enumerate(examples['ner_tags'])
    ]
    return tokenized_inputs

In [ ]:
#| export
def seqeval_compute_metrics(label_list: list, metric=None):
    """
    Return a `compute_metrics` callback scoring NER predictions via seqeval.

    Strips `-100`-labelled positions, decodes ids through `label_list`, and
    delegates to `metric` (default: `evaluate.load('seqeval')`, loaded lazily
    so importing this module never requires it).

    **Parameters**
    - `label_list` (*list*) — Id-to-label list, e.g. `['O', 'B-definition', ...]`.
    - `metric` — Object with `.compute(predictions=..., references=...)`; injectable for tests.
    """
    if metric is None:
        import evaluate
        metric = evaluate.load('seqeval')
    def compute_metrics(eval_preds):
        pred_ids = np.argmax(np.asarray(eval_preds.predictions), axis=2)
        predictions = [[label_list[p] for (p, l) in zip(row, lab) if l != -100]
                       for row, lab in zip(pred_ids, eval_preds.label_ids)]
        references = [[label_list[l] for (p, l) in zip(row, lab) if l != -100]
                      for row, lab in zip(pred_ids, eval_preds.label_ids)]
        return metric.compute(predictions=predictions, references=references)
    return compute_metrics

In [ ]:
#| export
def classification_compute_metrics(threshold: float = 0.5):
    """
    Return a `compute_metrics` callback for binary (logit, label) pairs.

    Applies a sigmoid and `threshold`, then reports accuracy, precision,
    recall, and F1 — the evaluation shared by the pair/linking classifiers,
    in pure numpy so it needs no metric packages.

    **Parameters**
    - `threshold` (*float*) — Sigmoid cut-off for a positive prediction.
    """
    def compute_metrics(eval_preds):
        probs = 1 / (1 + np.exp(-np.asarray(eval_preds.predictions, dtype=float)))
        preds = (probs >= threshold).astype(int).ravel()
        labels = np.asarray(eval_preds.label_ids).ravel().astype(int)
        tp = int(((preds == 1) & (labels == 1)).sum())
        fp = int(((preds == 1) & (labels == 0)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        return {'accuracy': float((preds == labels).mean()),
                'precision': precision, 'recall': recall, 'f1': f1}
    return compute_metrics

In [ ]:
#| export
def _prf(tp: int, n_pred: int, n_gold: int) -> dict:
    """Return a precision/recall/F1/counts dict; zero denominators score 0.0."""
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    return {'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p + r else 0.0,
            'pred': n_pred, 'gold': n_gold}


def span_f1_metrics(pred_rows: list, gold_rows: list,
                    labels: tuple = ('definition', 'notation')) -> dict:
    """
    Micro and per-label precision/recall/F1 over exact-match spans.

    Each element of `pred_rows` and `gold_rows` is a list of span dicts with
    `start`, `end`, and `label` keys (extra keys such as `verified` are
    ignored); the two lists correspond row-by-row. A predicted span scores a
    true positive only when start, end, and label all match a gold span.

    **Parameters**
    - `pred_rows` (`list`) — per row, the predicted span dicts.
    - `gold_rows` (`list`) — per row, the gold span dicts.
    - `labels` (`tuple`) — labels always reported, even when never observed;
      other labels seen in the data are added automatically.
    """
    keys = lambda spans: {(s['start'], s['end'], s['label']) for s in spans}
    stats = {lab: [0, 0, 0] for lab in labels}
    for preds, golds in zip((keys(r) for r in pred_rows), (keys(r) for r in gold_rows)):
        for lab in sorted({s[2] for s in preds | golds} - set(stats)):
            stats[lab] = [0, 0, 0]
        for lab in stats:
            tp_p = {s for s in preds if s[2] == lab}
            tp_g = {s for s in golds if s[2] == lab}
            stats[lab][0] += len(tp_p & tp_g)
            stats[lab][1] += len(tp_p)
            stats[lab][2] += len(tp_g)
    out = {lab: _prf(*st) for lab, st in stats.items()}
    out['micro'] = _prf(*(sum(st[i] for st in stats.values()) for i in range(3)))
    return out


In [ ]:
#| export
def load_math_tokens(tokenizer, token_file: PathLike) -> int:
    """
    Add the LaTeX command vocabulary from `token_file` to `tokenizer`.

    The file holds one token per line (e.g. `\\dim`) — the vocabulary
    generated once from the math corpus and reused by every training run,
    so all runs share identical embeddings indices. Callers that also load
    a model must follow up with `model.resize_token_embeddings(len(tokenizer))`.

    **Parameters**
    - `tokenizer` — HF tokenizer to extend.
    - `token_file` (*PathLike*) — Newline-separated token list.

    **Returns**
    Number of tokens actually added (already-known tokens are skipped by the tokenizer).
    """
    tokens = [ln.strip() for ln in Path(token_file).read_text(encoding='utf-8').splitlines() if ln.strip()]
    return tokenizer.add_tokens(tokens)

In [ ]:
import tempfile
from types import SimpleNamespace
from unittest import mock

from fastcore.test import *

In [ ]:
#| hide
# Pure alignment: special tokens (None) -> -100; continuation subwords repeat or -100.
word_ids = [None, 0, 0, 1, None]
test_eq(_aligned_label_ids(word_ids, [3, 4]), [-100, 3, 3, 4, -100])
test_eq(_aligned_label_ids(word_ids, [3, 4], label_all_tokens=False), [-100, 3, -100, 4, -100])

# End-to-end through a fake fast tokenizer whose word_ids mapping is fixed.
class _FakeEncoding(dict):
    def __init__(self, wids): super().__init__(input_ids=[[1]]); self._w = wids
    def word_ids(self, batch_index): return self._w[batch_index]
class _FakeTokenizer:
    def __init__(self, wids): self._w = wids
    def __call__(self, texts, **kwargs): return _FakeEncoding(self._w)

examples = {'tokens': [['dim', 'M']], 'ner_tags': [[1, 0]]}
out = align_labels_to_tokens(examples, _FakeTokenizer([word_ids]))
test_eq(out['labels'], [[-100, 1, 1, 0, -100]])

In [ ]:
#| hide
# seqeval factory decodes ids and strips -100 positions before delegating.
fake_metric = mock.MagicMock(return_value={'f1': 1.0})
label_list = ['O', 'B-definition', 'I-definition']
compute_metrics = seqeval_compute_metrics(label_list, metric=fake_metric)
# One row, three tokens: logits pick ids [1, 2, 0]; labels [1, 2, -100].
logits = np.zeros((1, 3, 3)); logits[0, 0, 1] = logits[0, 1, 2] = 1.0
eval_preds = SimpleNamespace(predictions=logits, label_ids=np.array([[1, 2, -100]]))
compute_metrics(eval_preds)
kwargs = fake_metric.compute.call_args.kwargs
test_eq(kwargs['predictions'], [['B-definition', 'I-definition']])
test_eq(kwargs['references'], [['B-definition', 'I-definition']])

# Binary classifier metrics on a hand-checkable confusion matrix.
compute_metrics = classification_compute_metrics(threshold=0.5)
# logits: strong+, weak+ (below cut), strong-, weak- -> preds 1,0,0,1 vs labels 1,1,0,0
logits = np.array([[3.0], [-0.5], [-3.0], [0.5]])
labels = np.array([1, 1, 0, 0])
res = compute_metrics(SimpleNamespace(predictions=logits, label_ids=labels))
test_eq(res['accuracy'], 0.5)   # tp=1 (row 0), tn=1 (row 2), fp=1 (row 3), fn=1 (row 1)
test_eq(res['precision'], 0.5)
test_eq(res['recall'], 0.5)
test_eq(res['f1'], 0.5)

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as tmp:
    token_file = Path(tmp) / 'latex_command_tokens.txt'
    token_file.write_text('\\dim\n\\to\n\n', encoding='utf-8')
    fake_tokenizer = mock.MagicMock()
    fake_tokenizer.add_tokens.return_value = 2
    test_eq(load_math_tokens(fake_tokenizer, token_file), 2)
    fake_tokenizer.add_tokens.assert_called_once_with(['\\dim', '\\to'])

In [ ]:
from fastcore.test import test_close, test_eq

pred_rows = [[{'start': 0, 'end': 5, 'label': 'definition', 'verified': True},
              {'start': 10, 'end': 12, 'label': 'notation'}],
             [{'start': 3, 'end': 4, 'label': 'notation'}]]
gold_rows = [[{'start': 0, 'end': 5, 'label': 'definition'},
              {'start': 9, 'end': 12, 'label': 'notation'}],
             []]
m = span_f1_metrics(pred_rows, gold_rows)
test_eq(m['definition'], {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'pred': 1, 'gold': 1})
test_eq((m['notation']['pred'], m['notation']['gold']), (2, 1))
test_close(m['notation']['precision'], 0.0)  # exact match: the off-by-one span is a miss
test_eq((m['micro']['pred'], m['micro']['gold']), (3, 2))
test_close(m['micro']['f1'], 2 * (1 / 3) * 0.5 / (1 / 3 + 0.5))
# unseen default labels report zeros instead of missing; empty input still scores
empty = span_f1_metrics([], [])
test_eq((empty['definition']['f1'], empty['notation']['f1']), (0.0, 0.0))
extra = span_f1_metrics([[{'start': 0, 'end': 1, 'label': 'split'}]],
                        [[{'start': 0, 'end': 1, 'label': 'split'}]])
test_eq(extra['split']['f1'], 1.0)
